# Lonestar Post-Flight Post-Mortem

This notebook is a compact diagnostic for the `Lonestar_2026_Itzamina` flight.

It will:
- rebuild the aligned GPS + MARV merged log if it does not exist
- summarize alignment, timing, and trajectory metrics
- report extrema for acceleration, gyro, baro, and GPS velocity channels
- derive a flight-stage timeline suitable for phase-specific process-model work
- flag likely clipping / saturation behavior from repeated exact extremes
- render the main matplotlib plots inline for post-mortem review

In [ ]:
%matplotlib inline

from __future__ import annotations

import importlib.util
import json
from pathlib import Path
import sys

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root from current working directory")


def load_module(path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load module from {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def first_true_time(frame: pd.DataFrame, column: str) -> float | None:
    if column not in frame.columns:
        return None
    flags = frame[column].astype(str).str.lower().isin({"true", "1"})
    if not flags.any():
        return None
    return float(pd.to_numeric(frame.loc[flags, "time_s"], errors="coerce").dropna().iloc[0])


def summarize_extrema(frame: pd.DataFrame, time_column: str, columns: dict[str, str]) -> pd.DataFrame:
    rows = []
    time_s = pd.to_numeric(frame[time_column], errors="coerce")
    for label, column in columns.items():
        series = pd.to_numeric(frame[column], errors="coerce")
        min_idx = series.idxmin()
        max_idx = series.idxmax()
        rows.append(
            {
                "channel": label,
                "min_value": float(series.loc[min_idx]),
                "min_time_s": float(time_s.loc[min_idx]),
                "max_value": float(series.loc[max_idx]),
                "max_time_s": float(time_s.loc[max_idx]),
            }
        )
    return pd.DataFrame(rows)


def repeated_extreme_report(series: pd.Series, *, tolerance: float = 1e-9) -> dict[str, float]:
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if values.size == 0:
        return {
            "sample_count": 0,
            "max_value": np.nan,
            "min_value": np.nan,
            "max_repeat_count": 0,
            "min_repeat_count": 0,
            "max_repeat_fraction": 0.0,
            "min_repeat_fraction": 0.0,
        }
    max_value = float(np.nanmax(values))
    min_value = float(np.nanmin(values))
    max_repeat_count = int(np.count_nonzero(np.isclose(values, max_value, atol=tolerance, rtol=0.0)))
    min_repeat_count = int(np.count_nonzero(np.isclose(values, min_value, atol=tolerance, rtol=0.0)))
    sample_count = int(values.size)
    return {
        "sample_count": sample_count,
        "max_value": max_value,
        "min_value": min_value,
        "max_repeat_count": max_repeat_count,
        "min_repeat_count": min_repeat_count,
        "max_repeat_fraction": max_repeat_count / sample_count,
        "min_repeat_fraction": min_repeat_count / sample_count,
    }


def detect_flight_stages(
    time_s: np.ndarray,
    accel_norm_mps2: np.ndarray,
    vertical_velocity_mps: np.ndarray,
    *,
    landing_time_s: float | None,
    gravity_magnitude_mps2: float = 9.80665,
    powered_accel_excess_mps2: float = 15.0,
    burnout_accel_hysteresis_mps2: float = 5.0,
    descent_velocity_threshold_mps: float = 2.0,
) -> np.ndarray:
    """Mirror the repo flight-phase detector semantics without importing the adapter package.

    Stages:
    - ON_PAD
    - BOOST
    - COAST
    - DESCENT
    - LANDED
    """

    stage_labels = []
    launched = False
    phase = "ON_PAD"
    for t, accel_norm, vertical_velocity in zip(time_s, accel_norm_mps2, vertical_velocity_mps):
        excess_acceleration = abs(float(accel_norm) - float(gravity_magnitude_mps2))

        if landing_time_s is not None and float(t) >= float(landing_time_s):
            phase = "LANDED"
            stage_labels.append(phase)
            continue

        if not launched:
            if excess_acceleration > float(powered_accel_excess_mps2):
                launched = True
                phase = "BOOST"
            else:
                phase = "ON_PAD"
            stage_labels.append(phase)
            continue

        if phase == "BOOST":
            if excess_acceleration < float(burnout_accel_hysteresis_mps2):
                phase = "COAST"
            stage_labels.append(phase)
            continue

        if phase == "COAST":
            if excess_acceleration > float(powered_accel_excess_mps2):
                phase = "BOOST"
            elif float(vertical_velocity) < -float(descent_velocity_threshold_mps):
                phase = "DESCENT"
            stage_labels.append(phase)
            continue

        if phase == "DESCENT":
            if excess_acceleration > float(powered_accel_excess_mps2):
                phase = "BOOST"
            stage_labels.append(phase)
            continue

        stage_labels.append(phase)

    return np.asarray(stage_labels, dtype=object)


def build_stage_segments(frame: pd.DataFrame, *, stage_column: str = "flight_stage") -> pd.DataFrame:
    rows = []
    if frame.empty:
        return pd.DataFrame(columns=["stage", "start_time_s", "end_time_s", "duration_s", "sample_count"])

    start_index = 0
    current_stage = str(frame.iloc[0][stage_column])
    for index in range(1, len(frame)):
        stage = str(frame.iloc[index][stage_column])
        if stage == current_stage:
            continue
        segment = frame.iloc[start_index:index]
        rows.append(
            {
                "stage": current_stage,
                "start_time_s": float(segment["time_s"].iloc[0]),
                "end_time_s": float(segment["time_s"].iloc[-1]),
                "duration_s": float(segment["time_s"].iloc[-1] - segment["time_s"].iloc[0]),
                "sample_count": int(len(segment)),
                "max_altitude_m": float(pd.to_numeric(segment["featherweight_navigation_gps_altitude_agl_m"], errors="coerce").max()),
                "max_accel_norm_mps2": float(pd.to_numeric(segment["marv_primary_imu_accel_norm_mps2"], errors="coerce").max()),
                "max_gyro_norm_rad_s": float(pd.to_numeric(segment["marv_primary_imu_gyro_norm_rad_s"], errors="coerce").max()),
                "mean_vertical_speed_mps": float(pd.to_numeric(segment["featherweight_navigation_vertical_speed_mps"], errors="coerce").mean()),
            }
        )
        start_index = index
        current_stage = stage

    segment = frame.iloc[start_index:]
    rows.append(
        {
            "stage": current_stage,
            "start_time_s": float(segment["time_s"].iloc[0]),
            "end_time_s": float(segment["time_s"].iloc[-1]),
            "duration_s": float(segment["time_s"].iloc[-1] - segment["time_s"].iloc[0]),
            "sample_count": int(len(segment)),
            "max_altitude_m": float(pd.to_numeric(segment["featherweight_navigation_gps_altitude_agl_m"], errors="coerce").max()),
            "max_accel_norm_mps2": float(pd.to_numeric(segment["marv_primary_imu_accel_norm_mps2"], errors="coerce").max()),
            "max_gyro_norm_rad_s": float(pd.to_numeric(segment["marv_primary_imu_gyro_norm_rad_s"], errors="coerce").max()),
            "mean_vertical_speed_mps": float(pd.to_numeric(segment["featherweight_navigation_vertical_speed_mps"], errors="coerce").mean()),
        }
    )
    return pd.DataFrame(rows)


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
repo_root

In [ ]:
dataset_dir = repo_root / "data" / "Lonestar_2026_Itzamina"
postflight_dir = dataset_dir / "postflight"
alignment_dir = postflight_dir / "alignment_gps_marv"

gps_path = dataset_dir / "UTRGV GPS _03-29-2026_08_47_14.csv"
marv_path = dataset_dir / "FLGT0023_TRIMMED.CSV"
merged_path = postflight_dir / "merged_gps_marv_primary_imu.csv"
merged_summary_path = merged_path.with_suffix(".json")
alignment_report_path = alignment_dir / "gps_baro_alignment_report.json"
rocket_config_path = repo_root / "notebooks" / "Itzamna" / "rocket_config.py"

boost_window_s = 5.0
timebase_source = "marv_primary_imu"

pd.Series(
    {
        "gps": gps_path,
        "marv": marv_path,
        "merged": merged_path,
        "alignment_report": alignment_report_path,
        "rocket_config": rocket_config_path,
    },
    dtype=object,
)

In [ ]:
lonestar = load_module(repo_root / "tools" / "lonestar_telemetry.py", "lonestar_telemetry_notebook")

if not merged_path.exists():
    postflight_dir.mkdir(parents=True, exist_ok=True)
    gps_source = lonestar.normalize_featherweight_gps_csv(gps_path)
    marv_source = lonestar.normalize_marv_csv(marv_path)

    alignment_report, gps_debug, marv_debug = lonestar.align_gps_altitude_to_marv_baro(
        gps_source,
        marv_source,
    )
    lonestar.write_gps_baro_alignment_artifacts(
        gps_source=gps_source,
        marv_source=marv_source,
        alignment_report=alignment_report,
        gps_debug=gps_debug,
        marv_debug=marv_debug,
        output_dir=alignment_dir,
    )

    merged, merge_summary = lonestar.merge_aligned_sources(
        gps_source,
        marv_source,
        offset_s=float(alignment_report["offset_s"]),
        timebase_source=timebase_source,
    )
    merged.to_csv(merged_path, index=False)
    merge_summary = {
        **merge_summary,
        "output_csv": str(merged_path),
        "alignment_context": {
            "source": "alignment_report",
            "report_path": str(alignment_report_path),
            "confidence": alignment_report.get("confidence"),
            "status": alignment_report.get("status"),
            "strong_match": alignment_report.get("strong_match"),
            "warnings": alignment_report.get("warnings", []),
        },
    }
    merged_summary_path.write_text(json.dumps(merge_summary, indent=2), encoding="utf-8")
    print(f"Created merged log: {merged_path}")
else:
    print(f"Using existing merged log: {merged_path}")

In [ ]:
plot_mod = load_module(repo_root / "tools" / "plot_lonestar_postflight.py", "plot_lonestar_postflight_notebook")

merged = plot_mod.load_merged_log(merged_path).copy()
merged_summary = json.loads(merged_summary_path.read_text(encoding="utf-8")) if merged_summary_path.exists() else {}
alignment_report = json.loads(alignment_report_path.read_text(encoding="utf-8")) if alignment_report_path.exists() else {}
alignment = alignment_report.get("alignment", {}) if isinstance(alignment_report, dict) else {}
alignment_context = merged_summary.get("alignment_context", {})

dry_mass_kg, motor_overlay = plot_mod._load_motor_overlay(rocket_config_path)

trajectory, reference_summary = plot_mod.build_trajectory_frame(
    merged,
    reference_latitude_deg=None,
    reference_longitude_deg=None,
)
altitude_frame = plot_mod.build_altitude_frame(merged)
thrust_frame, thrust_summary = plot_mod.estimate_apparent_thrust_curve(
    merged,
    dry_mass_kg=dry_mass_kg,
    motor_overlay=motor_overlay,
    boost_window_s=boost_window_s,
)

accelerometer_columns = {
    "Accel X": "marv_primary_imu_accelerometer_x_mps2",
    "Accel Y": "marv_primary_imu_accelerometer_y_mps2",
    "Accel Z": "marv_primary_imu_accelerometer_z_mps2",
}
gyroscope_columns = {
    "Gyro X": "marv_primary_imu_gyroscope_x_rad_s",
    "Gyro Y": "marv_primary_imu_gyroscope_y_rad_s",
    "Gyro Z": "marv_primary_imu_gyroscope_z_rad_s",
}
gps_velocity_columns = {
    "GPS Horizontal Speed": "featherweight_navigation_horizontal_speed_mps",
    "GPS Vertical Speed": "featherweight_navigation_vertical_speed_mps",
}

merged["marv_primary_imu_accel_norm_mps2"] = np.sqrt(
    pd.to_numeric(merged["marv_primary_imu_accelerometer_x_mps2"], errors="coerce") ** 2
    + pd.to_numeric(merged["marv_primary_imu_accelerometer_y_mps2"], errors="coerce") ** 2
    + pd.to_numeric(merged["marv_primary_imu_accelerometer_z_mps2"], errors="coerce") ** 2
)
merged["marv_primary_imu_gyro_norm_rad_s"] = np.sqrt(
    pd.to_numeric(merged["marv_primary_imu_gyroscope_x_rad_s"], errors="coerce") ** 2
    + pd.to_numeric(merged["marv_primary_imu_gyroscope_y_rad_s"], errors="coerce") ** 2
    + pd.to_numeric(merged["marv_primary_imu_gyroscope_z_rad_s"], errors="coerce") ** 2
)
merged["featherweight_navigation_gps_speed_norm_mps"] = np.sqrt(
    pd.to_numeric(merged["featherweight_navigation_horizontal_speed_mps"], errors="coerce") ** 2
    + pd.to_numeric(merged["featherweight_navigation_vertical_speed_mps"], errors="coerce") ** 2
)
if "marv_baro_altitude_rel_m" in altitude_frame:
    merged["marv_baro_altitude_rel_m"] = pd.to_numeric(altitude_frame["marv_baro_altitude_rel_m"], errors="coerce")

launch_time_gps_s = first_true_time(merged, "featherweight_events_launch_detected")
apogee_time_gps_s = first_true_time(merged, "featherweight_events_apogee_detected")
landing_time_gps_s = first_true_time(merged, "featherweight_events_landing_detected")

stage_colors = {
    "ON_PAD": "#7d8597",
    "BOOST": "#d62828",
    "COAST": "#fcbf49",
    "DESCENT": "#0077b6",
    "LANDED": "#2d6a4f",
}
stage_model_hints = pd.DataFrame(
    [
        {"stage": "ON_PAD", "process_model_hint": "near-static / bias characterization", "measurement_hint": "high trust in baro and gravity alignment"},
        {"stage": "BOOST", "process_model_hint": "high-thrust, rapidly varying specific force", "measurement_hint": "disable gravity updates; trust gyros and high-rate IMU"},
        {"stage": "COAST", "process_model_hint": "ballistic ascent / low-thrust free flight", "measurement_hint": "gravity alignment becomes valid again; baro + GPS useful"},
        {"stage": "DESCENT", "process_model_hint": "drag/parachute dominated descent", "measurement_hint": "vertical process noise should increase; GPS/baro dominate"},
        {"stage": "LANDED", "process_model_hint": "static landed state", "measurement_hint": "zero-velocity and bias locking opportunities"},
    ]
)

time_s = pd.to_numeric(merged["time_s"], errors="coerce").to_numpy(dtype=float)
accel_norm_mps2 = pd.to_numeric(merged["marv_primary_imu_accel_norm_mps2"], errors="coerce").to_numpy(dtype=float)
vertical_velocity_mps = pd.to_numeric(merged["featherweight_navigation_vertical_speed_mps"], errors="coerce").fillna(method="ffill").fillna(method="bfill").to_numpy(dtype=float)
merged["flight_stage"] = detect_flight_stages(
    time_s,
    accel_norm_mps2,
    vertical_velocity_mps,
    landing_time_s=landing_time_gps_s,
)
stage_id_map = {stage: index for index, stage in enumerate(["ON_PAD", "BOOST", "COAST", "DESCENT", "LANDED"])}
merged["flight_stage_id"] = merged["flight_stage"].map(stage_id_map)
stage_segments = build_stage_segments(merged)

apogee_idx = int(trajectory["up_m"].idxmax())
downrange = np.sqrt(trajectory["east_m"] ** 2 + trajectory["north_m"] ** 2)
max_downrange_idx = int(downrange.idxmax())
launch_time_s = float(thrust_summary["launch_time_s"])
gyro_boost_frame = merged.loc[
    (pd.to_numeric(merged["time_s"], errors="coerce") >= launch_time_s)
    & (pd.to_numeric(merged["time_s"], errors="coerce") <= launch_time_s + boost_window_s)
].copy()
gyro_boost_frame["time_since_launch_s"] = pd.to_numeric(gyro_boost_frame["time_s"], errors="coerce") - launch_time_s

overview_table = pd.DataFrame(
    [
        {"metric": "Merged rows", "value": int(len(merged))},
        {"metric": "Merged time start (s)", "value": float(merged["time_s"].iloc[0])},
        {"metric": "Merged time end (s)", "value": float(merged["time_s"].iloc[-1])},
        {"metric": "Timebase source", "value": merged_summary.get("timebase", {}).get("source")},
        {"metric": "MARV primary IMU approx rate (Hz)", "value": merged_summary.get("timebase", {}).get("approx_rate_hz")},
        {"metric": "GPS launch event time (s)", "value": launch_time_gps_s},
        {"metric": "GPS apogee event time (s)", "value": apogee_time_gps_s},
        {"metric": "GPS landing event time (s)", "value": landing_time_gps_s},
        {"metric": "Derived boost start (s)", "value": stage_segments.loc[stage_segments["stage"] == "BOOST", "start_time_s"].iloc[0] if (stage_segments["stage"] == "BOOST").any() else None},
        {"metric": "Derived descent start (s)", "value": stage_segments.loc[stage_segments["stage"] == "DESCENT", "start_time_s"].iloc[0] if (stage_segments["stage"] == "DESCENT").any() else None},
        {"metric": "Max GPS altitude AGL (m)", "value": float(trajectory.loc[apogee_idx, "up_m"])},
        {"metric": "Max GPS altitude time (s)", "value": float(trajectory.loc[apogee_idx, "time_s"])},
        {"metric": "Max downrange extent (m)", "value": float(downrange.loc[max_downrange_idx])},
        {"metric": "Time of max downrange (s)", "value": float(trajectory.loc[max_downrange_idx, "time_s"])},
    ]
)

alignment_table = pd.DataFrame(
    [
        {"metric": "Alignment offset (s)", "value": alignment.get("offset_s")},
        {"metric": "Alignment status", "value": alignment.get("status")},
        {"metric": "Strong match", "value": alignment.get("strong_match")},
        {"metric": "Confidence", "value": alignment.get("confidence")},
        {"metric": "Altitude correlation", "value": alignment.get("altitude_correlation")},
        {"metric": "Derivative correlation", "value": alignment.get("derivative_correlation")},
        {"metric": "Altitude RMSE (m)", "value": alignment.get("rmse_m")},
    ]
)

full_accel_extrema = summarize_extrema(merged, "time_s", {**accelerometer_columns, "Accel Norm": "marv_primary_imu_accel_norm_mps2"})
boost_accel_extrema = summarize_extrema(thrust_frame, "time_since_launch_s", {"Accel X": "accelerometer_x_mps2", "Accel Y": "accelerometer_y_mps2", "Accel Z": "accelerometer_z_mps2", "Accel Norm": "accelerometer_norm_mps2"})
full_gyro_extrema = summarize_extrema(merged, "time_s", {**gyroscope_columns, "Gyro Norm": "marv_primary_imu_gyro_norm_rad_s"})
boost_gyro_extrema = summarize_extrema(gyro_boost_frame, "time_since_launch_s", {"Gyro X": "marv_primary_imu_gyroscope_x_rad_s", "Gyro Y": "marv_primary_imu_gyroscope_y_rad_s", "Gyro Z": "marv_primary_imu_gyroscope_z_rad_s", "Gyro Norm": "marv_primary_imu_gyro_norm_rad_s"})
gps_velocity_extrema = summarize_extrema(merged, "time_s", {**gps_velocity_columns, "GPS Speed Norm": "featherweight_navigation_gps_speed_norm_mps"})
baro_extrema = summarize_extrema(merged, "time_s", {"Baro Pressure": "marv_baro_pressure_pa", "Baro Relative Altitude": "marv_baro_altitude_rel_m"})

quality_rows = []
for label, column in {**accelerometer_columns, **gyroscope_columns}.items():
    report = repeated_extreme_report(merged[column])
    quality_rows.append({"channel": label, "min_value": report["min_value"], "min_repeat_count": report["min_repeat_count"], "min_repeat_fraction": report["min_repeat_fraction"], "max_value": report["max_value"], "max_repeat_count": report["max_repeat_count"], "max_repeat_fraction": report["max_repeat_fraction"]})
quality_table = pd.DataFrame(quality_rows)

postmortem_notes = []
suspicious = quality_table[(quality_table["max_repeat_count"] >= 10) | (quality_table["min_repeat_count"] >= 10)]
if not suspicious.empty:
    postmortem_notes.append("Repeated exact extreme values suggest possible clipping or saturation on: " + ", ".join(suspicious["channel"].tolist()) + ".")
if alignment.get("warnings"):
    postmortem_notes.extend(alignment.get("warnings", []))
if not postmortem_notes:
    postmortem_notes.append("No automated post-mortem warnings were triggered by the current notebook checks.")

## Flight Overview

In [ ]:
display(overview_table)

## Alignment Context

In [ ]:
display(alignment_table)
if alignment_context:
    display(pd.Series(alignment_context, dtype=object).to_frame(name="value"))

## Flight Stage Segmentation

In [ ]:
display(stage_segments)
display(stage_model_hints)

In [ ]:
figure, (ax_stage, ax_trace) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, height_ratios=[1, 2])
for _, row in stage_segments.iterrows():
    color = stage_colors.get(row['stage'], '#adb5bd')
    ax_stage.axvspan(row['start_time_s'], row['end_time_s'], color=color, alpha=0.9)
    midpoint = 0.5 * (row['start_time_s'] + row['end_time_s'])
    ax_stage.text(midpoint, 0.5, row['stage'], ha='center', va='center', fontsize=9)
ax_stage.set_ylim(0, 1)
ax_stage.set_yticks([])
ax_stage.set_title('Derived Flight Stage Timeline')

ax_trace.plot(merged['time_s'], pd.to_numeric(merged['featherweight_navigation_gps_altitude_agl_m'], errors='coerce'), color='#00798c', linewidth=2.0, label='GPS altitude AGL')
ax_trace.set_ylabel('Altitude (m)')
ax_trace.grid(True, alpha=0.3)
ax_vz = ax_trace.twinx()
ax_vz.plot(merged['time_s'], pd.to_numeric(merged['featherweight_navigation_vertical_speed_mps'], errors='coerce'), color='#d1495b', linewidth=1.3, alpha=0.85, label='GPS vertical speed')
ax_vz.set_ylabel('Vertical speed (m/s)')

for event_time, label, color in [(launch_time_gps_s, 'Launch', '#264653'), (apogee_time_gps_s, 'Apogee', '#f77f00'), (landing_time_gps_s, 'Landing', '#2d6a4f')]:
    if event_time is not None:
        ax_stage.axvline(event_time, color=color, linestyle='--', linewidth=1.0)
        ax_trace.axvline(event_time, color=color, linestyle='--', linewidth=1.0)

handles_a, labels_a = ax_trace.get_legend_handles_labels()
handles_b, labels_b = ax_vz.get_legend_handles_labels()
ax_trace.legend(handles_a + handles_b, labels_a + labels_b, loc='upper right')
ax_trace.set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## Accelerometer Extremes

In [ ]:
display(Markdown('### Full merged log extrema'))
display(full_accel_extrema)
display(Markdown(f'### Boost-window extrema (0 to {boost_window_s:.1f} s after boost start)'))
display(boost_accel_extrema)

## Gyroscope Extremes

In [ ]:
display(Markdown('### Full merged log extrema'))
display(full_gyro_extrema)
display(Markdown(f'### Boost-window extrema (0 to {boost_window_s:.1f} s after boost start)'))
display(boost_gyro_extrema)

## GPS Velocity and Baro Extremes

In [ ]:
display(Markdown('### GPS velocity extrema'))
display(gps_velocity_extrema)
display(Markdown('### MARV baro extrema'))
display(baro_extrema)

## Sensor Quality Flags

In [ ]:
display(quality_table)
display(pd.DataFrame({'postmortem_notes': postmortem_notes}))

## 3D GPS Trajectory

In [ ]:
figure = plt.figure(figsize=(10, 8))
axes = figure.add_subplot(111, projection='3d')
axes.plot(trajectory['east_m'], trajectory['north_m'], trajectory['up_m'], linewidth=2.0, color='#0b6e4f')
axes.scatter(trajectory['east_m'].iloc[0], trajectory['north_m'].iloc[0], trajectory['up_m'].iloc[0], color='#d1495b', s=40, label='Launch')
axes.scatter(trajectory['east_m'].iloc[apogee_idx], trajectory['north_m'].iloc[apogee_idx], trajectory['up_m'].iloc[apogee_idx], color='#edae49', s=50, label='Apogee')
axes.set_title('Lonestar 2026 Itzamna\n3D GPS Trajectory')
axes.set_xlabel('East (m)')
axes.set_ylabel('North (m)')
axes.set_zlabel('Altitude AGL (m)')
axes.legend(loc='upper left')
plt.show()

## Altitude Profile

In [ ]:
figure, axes = plt.subplots(figsize=(10, 5))
axes.plot(altitude_frame['time_s'], altitude_frame['gps_altitude_agl_m'], label='GPS altitude AGL', color='#00798c', linewidth=2.0)
if 'marv_baro_altitude_rel_m' in altitude_frame:
    axes.plot(altitude_frame['time_s'], altitude_frame['marv_baro_altitude_rel_m'], label='MARV baro altitude (relative)', color='#d1495b', linewidth=1.5, alpha=0.9)
if launch_time_gps_s is not None:
    axes.axvline(launch_time_gps_s, color='#264653', linestyle='--', alpha=0.6, label='Launch event')
if apogee_time_gps_s is not None:
    axes.axvline(apogee_time_gps_s, color='#edae49', linestyle='--', alpha=0.6, label='Apogee event')
axes.set_title('Lonestar 2026 Itzamna\nAltitude Profile')
axes.set_xlabel('Time (s)')
axes.set_ylabel('Altitude (m)')
axes.grid(True, alpha=0.3)
axes.legend()
plt.show()

## MARV Boost Acceleration Components

In [ ]:
figure, axes = plt.subplots(figsize=(10, 5))
axes.plot(thrust_frame['time_since_launch_s'], thrust_frame['accelerometer_x_mps2'], label='Accel X', color='#1d3557', linewidth=1.7)
axes.plot(thrust_frame['time_since_launch_s'], thrust_frame['accelerometer_y_mps2'], label='Accel Y', color='#2a9d8f', linewidth=1.7)
axes.plot(thrust_frame['time_since_launch_s'], thrust_frame['accelerometer_z_mps2'], label='Accel Z', color='#e76f51', linewidth=1.9)
axes.plot(thrust_frame['time_since_launch_s'], thrust_frame['accelerometer_norm_mps2'], label='Accel norm', color='#6d597a', linewidth=1.2, linestyle=':', alpha=0.8)
axes.set_title(f'Lonestar 2026 Itzamna\nMARV Boost Acceleration Components (0-{boost_window_s:.1f} s)')
axes.set_xlabel('Time Since Boost Start (s)')
axes.set_ylabel('Acceleration (m/s²)')
axes.set_xlim(0.0, boost_window_s)
axes.grid(True, alpha=0.3)
if 'nominal_motor_thrust_n' in thrust_frame:
    thrust_axes = axes.twinx()
    thrust_axes.plot(thrust_frame['time_since_launch_s'], thrust_frame['nominal_motor_thrust_n'], label='Nominal thrust', color='#264653', linewidth=1.2, linestyle='--', alpha=0.8)
    thrust_axes.set_ylabel('Nominal thrust (N)')
    handles, labels = axes.get_legend_handles_labels()
    thrust_handles, thrust_labels = thrust_axes.get_legend_handles_labels()
    axes.legend(handles + thrust_handles, labels + thrust_labels, loc='upper right')
else:
    axes.legend(loc='upper right')
peak_norm_row = boost_accel_extrema.loc[boost_accel_extrema['channel'] == 'Accel Norm'].iloc[0]
axes.scatter(peak_norm_row['max_time_s'], peak_norm_row['max_value'], color='#000000', s=30, zorder=5)
axes.annotate(f"Peak norm: {peak_norm_row['max_value']:.2f} m/s²", (peak_norm_row['max_time_s'], peak_norm_row['max_value']), textcoords='offset points', xytext=(8, 8))
plt.show()

## MARV Boost Gyroscope Components

## MARV Full-Flight Acceleration Components

In [ ]:
figure, axes = plt.subplots(figsize=(12, 5))
full_flight_time_s = pd.to_numeric(merged['time_s'], errors='coerce')
full_accel_x_mps2 = pd.to_numeric(merged['marv_primary_imu_accelerometer_x_mps2'], errors='coerce')
full_accel_y_mps2 = pd.to_numeric(merged['marv_primary_imu_accelerometer_y_mps2'], errors='coerce')
full_accel_z_mps2 = pd.to_numeric(merged['marv_primary_imu_accelerometer_z_mps2'], errors='coerce')
full_accel_norm_mps2 = pd.to_numeric(merged['marv_primary_imu_accel_norm_mps2'], errors='coerce')
axes.plot(full_flight_time_s, full_accel_x_mps2, label='Accel X', color='#1d3557', linewidth=1.4)
axes.plot(full_flight_time_s, full_accel_y_mps2, label='Accel Y', color='#2a9d8f', linewidth=1.4)
axes.plot(full_flight_time_s, full_accel_z_mps2, label='Accel Z', color='#e76f51', linewidth=1.6)
axes.plot(full_flight_time_s, full_accel_norm_mps2, label='Accel norm', color='#6d597a', linewidth=1.1, linestyle=':', alpha=0.85)
axes.set_title('Lonestar 2026 Itzamna\nMARV Full-Flight Acceleration Components')
axes.set_xlabel('Time (s)')
axes.set_ylabel('Acceleration (m/s²)')
axes.grid(True, alpha=0.3)
axes.legend(loc='upper right', ncols=2)
plt.show()

In [ ]:
figure, axes = plt.subplots(figsize=(10, 5))
axes.plot(gyro_boost_frame['time_since_launch_s'], gyro_boost_frame['marv_primary_imu_gyroscope_x_rad_s'], label='Gyro X', color='#003049', linewidth=1.7)
axes.plot(gyro_boost_frame['time_since_launch_s'], gyro_boost_frame['marv_primary_imu_gyroscope_y_rad_s'], label='Gyro Y', color='#669bbc', linewidth=1.7)
axes.plot(gyro_boost_frame['time_since_launch_s'], gyro_boost_frame['marv_primary_imu_gyroscope_z_rad_s'], label='Gyro Z', color='#c1121f', linewidth=1.9)
axes.plot(gyro_boost_frame['time_since_launch_s'], gyro_boost_frame['marv_primary_imu_gyro_norm_rad_s'], label='Gyro norm', color='#780000', linewidth=1.2, linestyle=':', alpha=0.8)
axes.set_title(f'Lonestar 2026 Itzamna\nMARV Boost Angular Rate Components (0-{boost_window_s:.1f} s)')
axes.set_xlabel('Time Since Boost Start (s)')
axes.set_ylabel('Angular Rate (rad/s)')
axes.set_xlim(0.0, boost_window_s)
axes.grid(True, alpha=0.3)
axes.legend(loc='upper right')
peak_gyro_row = boost_gyro_extrema.loc[boost_gyro_extrema['channel'] == 'Gyro Norm'].iloc[0]
axes.scatter(peak_gyro_row['max_time_s'], peak_gyro_row['max_value'], color='#000000', s=30, zorder=5)
axes.annotate(f"Peak gyro norm: {peak_gyro_row['max_value']:.2f} rad/s", (peak_gyro_row['max_time_s'], peak_gyro_row['max_value']), textcoords='offset points', xytext=(8, 8))
plt.show()